In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../../')

from dotenv import load_dotenv
load_dotenv()

In [ ]:
from samplepack_tools.audio.audio_file import read_samples, write_samples, read_concat_samples
from samplepack_tools.audio.features import extract_features
from samplepack_tools.audio.windowing import window_samples, window_audio_clip, apply_fade_to_clip
from samplepack_tools.grain_cloud import GrainCloud
from samplepack_tools.audio.audio_clip import AudioClip, clips_from_folder, clips_from_file_or_folder
from samplepack_tools.utils import get_files_of_types
import samplepack_tools.definitions as definitions
from samplepack_tools.manage_packs import update_samplepack_db
from samplepack_tools.audio.processors import most_dissimilar_segments, most_dissimilar_segments_clips, strip_silence, cross_fade_all, sparsify_by_mfccs
from samplepack_tools.audio import processors
import os

from copy import deepcopy

In [ ]:
directory = "/Users/carl/Downloads/cave-gypsy.wav"
clips = clips_from_file_or_folder(directory)

In [ ]:
prefix = "cave-gypsy"
outpath = f"/Users/carl/dev/tsvr-samplepack-tools/data/{prefix}"
window_size = 16384
hop_size = 16384

if not os.path.exists(outpath):
    os.makedirs(outpath)

index = 0

for c in clips:
    c.apply_processor(lambda x: strip_silence(x, db_thresh=-36))
    sparsified_clips = sparsify_by_mfccs(c.samples, top_k=200, window_size=16384, hop_size=16384, n_mfcc=28, return_clips=True)

    for i, sc in enumerate(sparsified_clips):
        clip = AudioClip.from_samples(sc, samplerate=c.samplerate)
        clip = apply_fade_to_clip(clip, fade_duration=0.01)
        clip.save(f"{outpath}/{prefix}_{index}.wav")
        index += 1